# Member 5 · Frozen SBERT + TextCNN
YU Guanlin · DSAI4205

普通 Jupyter Notebook。默认从已提交的逐条预测重算结果，不需要 GPU、嵌入缓存或云端账号。
重新训练需在最后一节明确开启；数据和固定分组保持团队版本。

In [ ]:
import sys, os, json, subprocess
from pathlib import Path
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/config.py').is_file()),None)
if ROOT is None:raise RuntimeError('请从仓库根目录或 Member5_file 内打开 Notebook')
sys.path.insert(0,str(ROOT));sys.path.insert(0,str(ROOT/'Member5_file'))
import pandas as pd
from IPython.display import display,Image,Markdown
from release import verify_evidence
from adopted_model import selected_version,default_evidence_directory
SELECTED=selected_version()
EVIDENCE=default_evidence_directory()
print('Project:',ROOT.name)

## 已完成的任务
冻结多语言 SBERT 生成句向量和 token 序列，后接独立 TextCNN；另有按折训练的 FastText 对照。
当前采用截图所指的第一轮：实际执行 C0/C1/C3/C7 四配置、分组嵌套验证及三个随机种子，完成 249 次拟合。保留集只输出预测，不评分。

In [ ]:
verification=verify_evidence(EVIDENCE)
print('Evidence verification passed')
print('Development rows:',verification['development_rows'],'| Reserved split rows:',verification['reserved_rows'])
print(verification['scope'])
display(pd.DataFrame(verification['original_mvp']))

## 当前采用版本：C3 v1
下表是本地实验的选模流程 OOF，并非固定最终模型的独立测试成绩。
当前采用版本的流程平均 F1=0.6167，同轮 C0=0.6070。三个种子方向为正，但区间跨零且 +0.0097 未达 +0.010 门槛；采用为工作版本，不宣称已证明优势。

In [ ]:
table=pd.read_csv(EVIDENCE/'comparison.csv')
display(table[['model','seed','accuracy','macro_precision','macro_recall','macro_f1']])
summary=json.loads((EVIDENCE/'conclusion.json').read_text())
display(pd.DataFrame([{'model':'Selection pipeline','mean_macro_f1':summary['selected_mean']},
                      {'model':'Same-protocol C0','mean_macro_f1':summary['control_mean']},
                      {'model':'Historical official baseline','mean_macro_f1':summary['baseline_macro_f1']}]))
print('Mean delta:',round(summary['mean_delta'],4),'| Practical target met:',summary['practical_target_met'])
display(pd.DataFrame(verification['class_tradeoffs']))

In [ ]:
display(pd.DataFrame([{k:SELECTED['model'][k] for k in ['candidate','filters','epochs','seed','parameters']}]))
print('0.6167 is a selection-pipeline estimate, not an independent score for fixed C3.')


In [ ]:
for name in ['confusion_selected.png','seed_comparison.png']:
    display(Image(filename=str(EVIDENCE/name)))

## 结果的边界
历史 OOF 和团队保留集曾被接触；这些是追加开发证据。训练／验证差距支持检查泛化问题，但不能唯一归因于模型容量或编码器。
上方核验重算公开预测和 bootstrap；历史完整训练、分组、源码及权重重载检查见 acceptance_forensics.json。
本发布版清除了云端执行代码，其新运行签名与历史训练版不同；不能把历史权重审计当成对修改后源码的重新训练证明。

## 可选：加载当前采用的原始 C3
保持 RUN_PREDICTION=False 即可无缓存查看全部结果。开启推理前按 README 接收原始 C3 权重和六个已审核数组，按哈希核对；不要重新编码后冒充原始输入。
此处只推理，不训练，也不按保留集标签评分。后续优化使用单独的新实验，既有六配置代码保留为研究入口。


In [ ]:
RUN_PREDICTION=False
if RUN_PREDICTION:
    from adopted_model import predict_cached,verify_team_archive
    saved=verify_team_archive()
    probabilities=predict_cached(saved['test_ids'].tolist(),device='cpu')
    display(pd.DataFrame(probabilities,columns=['Neutral','Positive','Negative']).assign(id=saved['test_ids']))
else:
    print('Review mode: no training, prediction or downloads were started.')
